# 🔬 DINO ViT Practice Notebook
## Understanding Images with a Self-Supervised Vision Transformer

In this notebook we will practice the following:

1. **Loading the DINO model** — Loading a pretrained ViT with the `timm` library
2. **Data collection** — Using real Dog/Cat images from the Oxford-IIIT Pet Dataset
3. **Linear Probing** — Frozen feature extractor + Linear classifier
4. **Cosine Similarity** — Comparing intra-class vs inter-class similarity
5. **t-SNE Visualization** — Visualizing the embedding space
6. **Attention Map Visualization** — Checking where the ViT looks

---

## 0. Environment Setup & Package Installation

In [ ]:
!pip install timm torch torchvision scikit-learn matplotlib numpy Pillow requests tqdm -q

In [ ]:
import os
import random
import requests
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from PIL import Image
from io import BytesIO
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader

import timm
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.manifold import TSNE
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🖥️ Using device: {device}")

---
## 1. Load the DINO ViT Model (timm)

DINO (Self-**DI**stillation with **NO** labels) is a self-supervised learning method
that showed a ViT trained without labels learns surprising visual features.

- Paper: [Emerging Properties in Self-Supervised Vision Transformers (Caron et al., 2021)](https://arxiv.org/abs/2104.14294)
- We use the `vit_small_patch16_224.dino` model from `timm`.

In [ ]:
# Load the ViT-Small model trained with DINO
model_name = 'vit_small_patch16_224.dino'
model = timm.create_model(model_name, pretrained=True, num_classes=0)  # num_classes=0 → feature extractor
model = model.to(device)
model.eval()

print(f"✅ Model loaded: {model_name}")
print(f"   - Patch size: 16x16")
print(f"   - Input size: 224x224")
print(f"   - Embedding dim: {model.embed_dim}")
print(f"   - Number of heads: {model.blocks[0].attn.num_heads}")
print(f"   - Number of layers: {len(model.blocks)}")

# Check the number of parameters
total_params = sum(p.numel() for p in model.parameters())
print(f"   - Total parameters: {total_params / 1e6:.1f}M")

### Image Preprocessing Pipeline

In [ ]:
# Use the model-specific transform provided by timm
data_config = timm.data.resolve_model_data_config(model)
transform = timm.data.create_transform(**data_config, is_training=False)

print("📐 Transform pipeline:")
print(transform)

---
## 2. Download Dog / Cat Images

We download real high-quality Dog/Cat images from the **Oxford-IIIT Pet Dataset**.
It is a widely used academic research dataset that provides high-quality images of 37 breeds.

- Source: [Oxford-IIIT Pet Dataset](https://www.robots.ox.ac.uk/~vgg/data/pets/)
- Dog/Cat is automatically classified by the filename convention.

> ⚠️ **Note**: About 800MB will be downloaded on first run. The cache is used afterwards.

In [ ]:
# Download the Oxford-IIIT Pet Dataset & classify into Dog/Cat
import tarfile
import shutil

NUM_IMAGES_PER_CLASS = 50  # Number of images per class (for practice; up to ~200 is possible)
IMG_DIR = './data'
CACHE_DIR = './cache'
os.makedirs(IMG_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(f'{IMG_DIR}/dog', exist_ok=True)
os.makedirs(f'{IMG_DIR}/cat', exist_ok=True)

# Cat breeds in Oxford-IIIT Pet (filenames starting with an uppercase letter are Cat)
# Rule: filenames starting with an uppercase letter are Cat, lowercase are Dog
# e.g., Abyssinian_100.jpg → Cat, beagle_100.jpg → Dog

def download_oxford_pets():
    """Download images from the Oxford-IIIT Pet Dataset."""
    tar_path = os.path.join(CACHE_DIR, 'images.tar.gz')
    extract_dir = os.path.join(CACHE_DIR, 'images')
    
    # Skip if already extracted
    if os.path.exists(extract_dir) and len(os.listdir(extract_dir)) > 100:
        print('   Using cached data.')
        return extract_dir
    
    # Download
    url = 'https://www.robots.ox.ac.uk/~vgg/data/pets/data/images.tar.gz'
    print(f'   Downloading: {url}')
    print(f'   (about 800MB, only required on first run)')
    
    response = requests.get(url, stream=True, timeout=120)
    total = int(response.headers.get('content-length', 0))
    
    with open(tar_path, 'wb') as f:
        downloaded = 0
        for chunk in response.iter_content(chunk_size=1024 * 1024):
            f.write(chunk)
            downloaded += len(chunk)
            if total:
                pct = downloaded / total * 100
                print(f'\r   Downloading: {downloaded/(1024*1024):.0f}MB / {total/(1024*1024):.0f}MB ({pct:.0f}%)', end='')
    print('\n   Download complete!')
    
    # Extract archive
    print('   Extracting archive...')
    with tarfile.open(tar_path, 'r:gz') as tar:
        tar.extractall(CACHE_DIR)
    
    # Images are inside the images/ folder
    inner_dir = os.path.join(CACHE_DIR, 'images')
    if os.path.exists(inner_dir):
        extract_dir = inner_dir
    
    print(f'   Extraction complete! ({len(os.listdir(extract_dir))} files)')
    return extract_dir

def organize_dog_cat(src_dir, dst_dir, num_per_class):
    """
    Classify Oxford-IIIT Pet images into Dog/Cat.
    Rule: filename starts with uppercase → Cat, starts with lowercase → Dog
    """
    dog_files = []
    cat_files = []
    
    for fname in sorted(os.listdir(src_dir)):
        if not fname.lower().endswith(('.jpg', '.jpeg', '.png')):
            continue
        # Oxford-IIIT Pet rule: starts with uppercase = Cat, starts with lowercase = Dog
        if fname[0].isupper():
            cat_files.append(fname)
        else:
            dog_files.append(fname)
    
    # Shuffle then select (so various breeds are mixed)
    random.shuffle(dog_files)
    random.shuffle(cat_files)
    
    dog_selected = dog_files[:num_per_class]
    cat_selected = cat_files[:num_per_class]
    
    # Copy
    for i, fname in enumerate(tqdm(dog_selected, desc='Copying dogs')):
        src = os.path.join(src_dir, fname)
        dst = os.path.join(dst_dir, 'dog', f'dog_{i:03d}.jpg')
        img = Image.open(src).convert('RGB')
        img.save(dst)
    
    for i, fname in enumerate(tqdm(cat_selected, desc='Copying cats')):
        src = os.path.join(src_dir, fname)
        dst = os.path.join(dst_dir, 'cat', f'cat_{i:03d}.jpg')
        img = Image.open(src).convert('RGB')
        img.save(dst)
    
    return len(dog_selected), len(cat_selected)

# Skip if data already exists
existing_dogs = len([f for f in os.listdir(f'{IMG_DIR}/dog') if f.endswith('.jpg')])
existing_cats = len([f for f in os.listdir(f'{IMG_DIR}/cat') if f.endswith('.jpg')])

if existing_dogs >= NUM_IMAGES_PER_CLASS and existing_cats >= NUM_IMAGES_PER_CLASS:
    print(f'✅ Data is already prepared: Dogs={existing_dogs}, Cats={existing_cats}')
else:
    print('📥 Starting Oxford-IIIT Pet Dataset download...\n')
    src_dir = download_oxford_pets()
    print(f'\n📂 Classifying into Dog/Cat...')
    n_dogs, n_cats = organize_dog_cat(src_dir, IMG_DIR, NUM_IMAGES_PER_CLASS)
    print(f'\n✅ Ready: Dogs={n_dogs}, Cats={n_cats}')

### Sample Image Visualization

In [ ]:
fig, axes = plt.subplots(2, 6, figsize=(18, 6))
fig.suptitle('📸 Sample Images from Dataset', fontsize=16, fontweight='bold')

for row, label in enumerate(['dog', 'cat']):
    img_files = sorted(os.listdir(f'{IMG_DIR}/{label}'))[:6]
    for col, fname in enumerate(img_files):
        img = Image.open(f'{IMG_DIR}/{label}/{fname}').convert('RGB')
        axes[row, col].imshow(img)
        axes[row, col].set_title(f'{label.upper()} #{col}', fontsize=11)
        axes[row, col].axis('off')

plt.tight_layout()
plt.show()

---
## 3. Feature Extraction & Linear Probing

### What is Linear Probing?
It is a method that **freezes** the weights of a pretrained model and
**only trains a Linear classifier** on top of the output features.

This approach lets you evaluate the quality of the pretrained representation.

```
Image → [Frozen DINO ViT] → CLS token (384-d) → [Trainable Linear Layer] → Dog/Cat
```

In [ ]:
class DogCatDataset(Dataset):
    """Dog vs Cat image dataset"""
    def __init__(self, img_dir, transform=None):
        self.samples = []
        self.transform = transform
        self.class_names = ['cat', 'dog']
        
        for label_idx, label in enumerate(self.class_names):
            class_dir = os.path.join(img_dir, label)
            if not os.path.exists(class_dir):
                continue
            for fname in sorted(os.listdir(class_dir)):
                if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
                    self.samples.append((os.path.join(class_dir, fname), label_idx))
        
        print(f"   Dataset: {len(self.samples)} images ({self.class_names})")
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, label, path

print("📂 Building dataset:")
dataset = DogCatDataset(IMG_DIR, transform=transform)

In [ ]:
@torch.no_grad()
def extract_features(model, dataset, device, batch_size=16):
    """
    Extract image features with DINO ViT.
    The CLS token output is used as the feature vector.
    """
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=0)
    
    all_features = []
    all_labels = []
    all_paths = []
    
    model.eval()
    for images, labels, paths in tqdm(loader, desc='Extracting features'):
        images = images.to(device)
        features = model(images)  # (B, embed_dim) — CLS token output
        
        all_features.append(features.cpu().numpy())
        all_labels.append(labels.numpy())
        all_paths.extend(paths)
    
    features = np.concatenate(all_features, axis=0)
    labels = np.concatenate(all_labels, axis=0)
    
    print(f"\n✅ Feature extraction complete!")
    print(f"   - Feature shape: {features.shape}")
    print(f"   - Labels shape: {labels.shape}")
    
    return features, labels, all_paths

features, labels, paths = extract_features(model, dataset, device)

In [ ]:
# Train / Test split
X_train, X_test, y_train, y_test, paths_train, paths_test = train_test_split(
    features, labels, paths,
    test_size=0.3,
    random_state=SEED,
    stratify=labels
)

print(f"📊 Train/Test Split:")
print(f"   Train: {len(X_train)} samples (Dog: {(y_train==1).sum()}, Cat: {(y_train==0).sum()})")
print(f"   Test:  {len(X_test)} samples (Dog: {(y_test==1).sum()}, Cat: {(y_test==0).sum()})")

In [ ]:
# Linear Probing: Logistic Regression (= 1-layer linear classifier)
print("🔧 Training linear probe...\n")

clf = LogisticRegression(
    max_iter=1000,
    random_state=SEED,
    C=1.0,          # regularization
    solver='lbfgs'
)
clf.fit(X_train, y_train)

# Evaluate
train_acc = clf.score(X_train, y_train)
test_acc = clf.score(X_test, y_test)
y_pred = clf.predict(X_test)

print(f"{'='*50}")
print(f"  📈 Linear Probing Results")
print(f"{'='*50}")
print(f"  Train Accuracy: {train_acc:.1%}")
print(f"  Test Accuracy:  {test_acc:.1%}")
print(f"{'='*50}\n")

print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Cat 🐱', 'Dog 🐶']))

### Confusion Matrix & Prediction Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# --- Confusion Matrix ---
cm = confusion_matrix(y_test, y_pred)
im = axes[0].imshow(cm, cmap='Blues')
axes[0].set_title('Confusion Matrix', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')
axes[0].set_xticks([0, 1])
axes[0].set_yticks([0, 1])
axes[0].set_xticklabels(['Cat', 'Dog'])
axes[0].set_yticklabels(['Cat', 'Dog'])
for i in range(2):
    for j in range(2):
        axes[0].text(j, i, str(cm[i, j]), ha='center', va='center', fontsize=20, fontweight='bold',
                    color='white' if cm[i, j] > cm.max()/2 else 'black')
plt.colorbar(im, ax=axes[0], shrink=0.8)

# --- Accuracy Bar Chart ---
bars = axes[1].bar(['Train', 'Test'], [train_acc, test_acc], 
                    color=['#4CAF50', '#2196F3'], width=0.5, edgecolor='black')
axes[1].set_ylim(0, 1.15)
axes[1].set_title('Linear Probing Accuracy', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Accuracy')
for bar, acc in zip(bars, [train_acc, test_acc]):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, 
                f'{acc:.1%}', ha='center', fontsize=16, fontweight='bold')
axes[1].axhline(y=0.5, color='red', linestyle='--', alpha=0.5, label='Random baseline')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Visualize predictions on test samples
n_show = 10
indices = random.sample(range(len(X_test)), min(n_show, len(X_test)))

fig, axes = plt.subplots(2, 5, figsize=(20, 8))
fig.suptitle('🔍 Test Set Predictions (Linear Probing)', fontsize=16, fontweight='bold')

class_names = ['Cat 🐱', 'Dog 🐶']

for i, idx in enumerate(indices):
    ax = axes[i // 5, i % 5]
    img = Image.open(paths_test[idx]).convert('RGB')
    ax.imshow(img)
    
    true_label = class_names[y_test[idx]]
    pred_label = class_names[y_pred[idx]]
    correct = y_test[idx] == y_pred[idx]
    
    color = 'green' if correct else 'red'
    symbol = '✓' if correct else '✗'
    ax.set_title(f'{symbol} Pred: {pred_label}\nTrue: {true_label}', 
                color=color, fontsize=11, fontweight='bold')
    ax.axis('off')

plt.tight_layout()
plt.show()

---
## 4. Cosine Similarity Analysis

We check whether, in the feature space learned by DINO, items in the same class have **high similarity**
and items in different classes have **low similarity**.

$$\text{cosine\_sim}(\mathbf{a}, \mathbf{b}) = \frac{\mathbf{a} \cdot \mathbf{b}}{\|\mathbf{a}\| \|\mathbf{b}\|}$$

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# Normalize features
features_norm = features / np.linalg.norm(features, axis=1, keepdims=True)

# Full cosine similarity matrix
cos_sim_matrix = cosine_similarity(features_norm)

# Separate by class
cat_idx = np.where(labels == 0)[0]
dog_idx = np.where(labels == 1)[0]

# Compute intra-class / inter-class similarity
cat_cat_sim = cos_sim_matrix[np.ix_(cat_idx, cat_idx)]
dog_dog_sim = cos_sim_matrix[np.ix_(dog_idx, dog_idx)]
cat_dog_sim = cos_sim_matrix[np.ix_(cat_idx, dog_idx)]

# Remove the diagonal (self)
cat_cat_vals = cat_cat_sim[np.triu_indices_from(cat_cat_sim, k=1)]
dog_dog_vals = dog_dog_sim[np.triu_indices_from(dog_dog_sim, k=1)]
cat_dog_vals = cat_dog_sim.flatten()

print(f"📊 Cosine Similarity Statistics:")
print(f"{'='*50}")
print(f"  Cat ↔ Cat  (intra-class):  {cat_cat_vals.mean():.4f} ± {cat_cat_vals.std():.4f}")
print(f"  Dog ↔ Dog  (intra-class):  {dog_dog_vals.mean():.4f} ± {dog_dog_vals.std():.4f}")
print(f"  Cat ↔ Dog  (inter-class):  {cat_dog_vals.mean():.4f} ± {cat_dog_vals.std():.4f}")
print(f"{'='*50}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# --- (1) Similarity histogram ---
axes[0].hist(cat_cat_vals, bins=30, alpha=0.6, label='Cat↔Cat', color='#FF6B6B', density=True)
axes[0].hist(dog_dog_vals, bins=30, alpha=0.6, label='Dog↔Dog', color='#4ECDC4', density=True)
axes[0].hist(cat_dog_vals, bins=30, alpha=0.6, label='Cat↔Dog', color='#95A5A6', density=True)
axes[0].set_xlabel('Cosine Similarity', fontsize=12)
axes[0].set_ylabel('Density', fontsize=12)
axes[0].set_title('Similarity Distribution', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].axvline(x=cat_dog_vals.mean(), color='gray', linestyle='--', alpha=0.7)

# --- (2) Similarity Heatmap (sorted) ---
sorted_idx = np.concatenate([cat_idx, dog_idx])
sorted_sim = cos_sim_matrix[np.ix_(sorted_idx, sorted_idx)]
im = axes[1].imshow(sorted_sim, cmap='RdYlBu_r', vmin=0, vmax=1)
axes[1].set_title('Cosine Similarity Matrix\n(sorted by class)', fontsize=14, fontweight='bold')
n_cats = len(cat_idx)
axes[1].axhline(y=n_cats-0.5, color='black', linewidth=2)
axes[1].axvline(x=n_cats-0.5, color='black', linewidth=2)
axes[1].set_xlabel('Sample Index')
axes[1].set_ylabel('Sample Index')
plt.colorbar(im, ax=axes[1], shrink=0.8)

# --- (3) Box Plot ---
bp = axes[2].boxplot(
    [cat_cat_vals, dog_dog_vals, cat_dog_vals],
    labels=['Cat↔Cat', 'Dog↔Dog', 'Cat↔Dog'],
    patch_artist=True,
    boxprops=dict(linewidth=2),
    medianprops=dict(linewidth=2, color='black')
)
colors = ['#FF6B6B', '#4ECDC4', '#95A5A6']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
axes[2].set_ylabel('Cosine Similarity', fontsize=12)
axes[2].set_title('Similarity Comparison', fontsize=14, fontweight='bold')
axes[2].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("\n💡 Observation: similarity within the same class is higher than across classes!")
print("   → Shows that DINO learned meaningful representations even without labels.")

### Pairwise Cosine Similarity Comparison Between Individual Images

In [ ]:
# Compare image pairs from the same class vs different classes
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('🔗 Pairwise Cosine Similarity Examples', fontsize=16, fontweight='bold')

pairs = [
    # (idx1, idx2, description)
    (cat_idx[0], cat_idx[1], 'Cat vs Cat'),
    (dog_idx[0], dog_idx[1], 'Dog vs Dog'),
    (cat_idx[0], dog_idx[0], 'Cat vs Dog'),
]

for col, (i, j, desc) in enumerate(pairs):
    sim = cos_sim_matrix[i, j]
    
    img1 = Image.open(paths[i]).convert('RGB')
    img2 = Image.open(paths[j]).convert('RGB')
    
    axes[0, col].imshow(img1)
    axes[0, col].set_title(f'Image A', fontsize=11)
    axes[0, col].axis('off')
    
    axes[1, col].imshow(img2)
    axes[1, col].set_title(f'Image B', fontsize=11)
    axes[1, col].axis('off')
    
    color = '#4CAF50' if sim > 0.7 else '#FF9800' if sim > 0.5 else '#F44336'
    fig.text(0.17 + col * 0.33, 0.48, f'{desc}\nSimilarity: {sim:.4f}', 
             ha='center', fontsize=14, fontweight='bold', color=color,
             bbox=dict(boxstyle='round,pad=0.5', facecolor='lightyellow', edgecolor=color))

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.subplots_adjust(hspace=0.4)
plt.show()

---
## 5. t-SNE Embedding Visualization

We reduce the high-dimensional (384-d) features to 2D for visualization.
t-SNE is a nonlinear method that reduces dimensionality while preserving relationships among nearby points.

If DINO's representation is good, you will see **clusters per class**.

In [ ]:
print("🔄 Computing t-SNE... (please wait a moment)")

tsne = TSNE(
    n_components=2,
    perplexity=min(30, len(features) - 1),
    random_state=SEED,
    n_iter=1000,
    learning_rate='auto',
    init='pca'
)

embeddings_2d = tsne.fit_transform(features_norm)
print(f"✅ t-SNE done! Shape: {embeddings_2d.shape}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# --- (1) Basic t-SNE plot ---
colors_map = {0: '#FF6B6B', 1: '#4ECDC4'}
markers_map = {0: 'o', 1: 's'}
label_map = {0: 'Cat 🐱', 1: 'Dog 🐶'}

for cls in [0, 1]:
    mask = labels == cls
    axes[0].scatter(
        embeddings_2d[mask, 0], embeddings_2d[mask, 1],
        c=colors_map[cls], marker=markers_map[cls],
        s=80, alpha=0.7, edgecolors='white', linewidth=0.5,
        label=label_map[cls]
    )
axes[0].set_title('t-SNE of DINO Features', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=13, loc='best')
axes[0].set_xlabel('t-SNE 1', fontsize=12)
axes[0].set_ylabel('t-SNE 2', fontsize=12)
axes[0].grid(alpha=0.2)

# --- (2) Show Train/Test split ---
# Map Train/Test indices
train_mask = np.zeros(len(labels), dtype=bool)
test_mask = np.zeros(len(labels), dtype=bool)

# Find train/test indices based on paths
paths_set_train = set(paths_train)
paths_set_test = set(paths_test)
for i, p in enumerate(paths):
    if p in paths_set_train:
        train_mask[i] = True
    elif p in paths_set_test:
        test_mask[i] = True

for cls in [0, 1]:
    # Train
    mask = (labels == cls) & train_mask
    axes[1].scatter(
        embeddings_2d[mask, 0], embeddings_2d[mask, 1],
        c=colors_map[cls], marker='o',
        s=60, alpha=0.5, edgecolors='white', linewidth=0.5,
        label=f'{label_map[cls]} (train)'
    )
    # Test
    mask = (labels == cls) & test_mask
    axes[1].scatter(
        embeddings_2d[mask, 0], embeddings_2d[mask, 1],
        c=colors_map[cls], marker='*',
        s=200, alpha=0.9, edgecolors='black', linewidth=1,
        label=f'{label_map[cls]} (test)'
    )

axes[1].set_title('t-SNE with Train/Test Split', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=10, loc='best')
axes[1].set_xlabel('t-SNE 1', fontsize=12)
axes[1].set_ylabel('t-SNE 2', fontsize=12)
axes[1].grid(alpha=0.2)

plt.tight_layout()
plt.show()

print("\n💡 Observation: Cat and Dog are well separated in t-SNE space!")
print("   → Self-supervised learning alone learns features useful for class separation.")

### Show Actual Image Thumbnails on the t-SNE Plot

In [ ]:
from matplotlib.offsetbox import OffsetImage, AnnotationBbox

fig, ax = plt.subplots(figsize=(14, 14))
ax.set_title('t-SNE with Image Thumbnails', fontsize=16, fontweight='bold')

# Scatter in the background
for cls in [0, 1]:
    mask = labels == cls
    ax.scatter(embeddings_2d[mask, 0], embeddings_2d[mask, 1],
              c=colors_map[cls], alpha=0.15, s=200)

# Show image thumbnails (only some)
n_thumbnails = min(30, len(paths))
thumb_indices = np.linspace(0, len(paths)-1, n_thumbnails, dtype=int)

for idx in thumb_indices:
    try:
        img = Image.open(paths[idx]).convert('RGB').resize((40, 40))
        img_arr = np.array(img)
        
        im = OffsetImage(img_arr, zoom=0.7)
        border_color = colors_map[labels[idx]]
        ab = AnnotationBbox(im, (embeddings_2d[idx, 0], embeddings_2d[idx, 1]),
                           frameon=True, pad=0.1,
                           bboxprops=dict(edgecolor=border_color, linewidth=2))
        ax.add_artist(ab)
    except:
        pass

ax.grid(alpha=0.2)
ax.set_xlabel('t-SNE 1', fontsize=12)
ax.set_ylabel('t-SNE 2', fontsize=12)
plt.tight_layout()
plt.show()

---
## 6. ViT Attention Map Visualization

DINO's key finding: **the attention of a self-supervised ViT naturally learns object segmentation!**

We visualize the attention weights that the CLS token assigns to other patches in the ViT's self-attention.

```
CLS token → Attention → [patch_1, patch_2, ..., patch_196]
                          ↓ reshape to 14×14
                          Attention Map!
```

In [ ]:
def get_attention_maps(model, img_tensor, device):
    """
    Extract attention maps from every layer of the ViT.
    
    Returns:
        attentions: list of (1, num_heads, num_tokens, num_tokens) tensors
    """
    model.eval()
    attentions = []
    hooks = []
    
    def hook_fn(module, input, output):
        # Capture the attention weight from timm's Attention module
        # Run the forward directly to obtain the attention weight
        B, N, C = input[0].shape
        qkv = module.qkv(input[0]).reshape(B, N, 3, module.num_heads, C // module.num_heads)
        qkv = qkv.permute(2, 0, 3, 1, 4)
        q, k, v = qkv.unbind(0)
        
        scale = (C // module.num_heads) ** -0.5
        attn = (q @ k.transpose(-2, -1)) * scale
        attn = attn.softmax(dim=-1)
        attentions.append(attn.detach().cpu())
    
    # Register a hook on every Attention module
    for block in model.blocks:
        hook = block.attn.register_forward_hook(hook_fn)
        hooks.append(hook)
    
    # Forward pass
    with torch.no_grad():
        _ = model(img_tensor.unsqueeze(0).to(device))
    
    # Remove hooks
    for hook in hooks:
        hook.remove()
    
    return attentions


def visualize_attention(img_path, model, transform, device, layer=-1):
    """
    Visualize the ViT attention map for an image.
    
    Args:
        layer: layer index to visualize (-1 = last layer)
    """
    # Load image
    img = Image.open(img_path).convert('RGB')
    img_tensor = transform(img)
    
    # Extract attention
    attentions = get_attention_maps(model, img_tensor, device)
    
    # Attention from the last layer (CLS token → patches)
    attn = attentions[layer]  # (1, num_heads, N, N)
    num_heads = attn.shape[1]
    
    # Attention from CLS token (index 0) to patch tokens
    cls_attn = attn[0, :, 0, 1:]  # (num_heads, num_patches)
    
    # Reshape to spatial map (14x14 for 224/16)
    patch_size = 16
    num_patches_side = 224 // patch_size  # 14
    cls_attn = cls_attn.reshape(num_heads, num_patches_side, num_patches_side)
    
    return img, cls_attn, num_heads

print("✅ Attention extraction function defined")

In [ ]:
# Visualize the attention map for a few sample images
sample_paths = []
for label in ['dog', 'cat']:
    files = sorted(os.listdir(f'{IMG_DIR}/{label}'))[:3]
    sample_paths.extend([f'{IMG_DIR}/{label}/{f}' for f in files])

for img_path in sample_paths[:4]:
    img, cls_attn, num_heads = visualize_attention(img_path, model, transform, device)
    
    # Mean over heads + per-head
    n_show_heads = min(6, num_heads)
    fig, axes = plt.subplots(2, n_show_heads + 1, figsize=(3 * (n_show_heads + 1), 6))
    
    label_name = 'Dog 🐶' if 'dog' in img_path else 'Cat 🐱'
    fig.suptitle(f'Attention Maps — {label_name} ({os.path.basename(img_path)})', 
                fontsize=14, fontweight='bold')
    
    # Row 1: Original + per-head attention (raw)
    img_resized = img.resize((224, 224))
    axes[0, 0].imshow(img_resized)
    axes[0, 0].set_title('Original', fontsize=10, fontweight='bold')
    axes[0, 0].axis('off')
    
    axes[1, 0].imshow(img_resized)
    # Mean attention overlay
    mean_attn = cls_attn.mean(0).numpy()
    mean_attn = (mean_attn - mean_attn.min()) / (mean_attn.max() - mean_attn.min() + 1e-8)
    mean_attn_resized = np.array(Image.fromarray((mean_attn * 255).astype(np.uint8)).resize((224, 224), Image.BILINEAR)) / 255.0
    axes[1, 0].imshow(mean_attn_resized, cmap='hot', alpha=0.6)
    axes[1, 0].set_title('Mean Attn\n(overlay)', fontsize=10, fontweight='bold')
    axes[1, 0].axis('off')
    
    for h in range(n_show_heads):
        attn_h = cls_attn[h].numpy()
        attn_h = (attn_h - attn_h.min()) / (attn_h.max() - attn_h.min() + 1e-8)
        attn_h_resized = np.array(Image.fromarray((attn_h * 255).astype(np.uint8)).resize((224, 224), Image.BILINEAR)) / 255.0
        
        # Raw attention
        axes[0, h+1].imshow(attn_h_resized, cmap='inferno')
        axes[0, h+1].set_title(f'Head {h}', fontsize=10)
        axes[0, h+1].axis('off')
        
        # Overlay on image
        axes[1, h+1].imshow(img_resized)
        axes[1, h+1].imshow(attn_h_resized, cmap='hot', alpha=0.6)
        axes[1, h+1].set_title(f'Head {h}\n(overlay)', fontsize=10)
        axes[1, h+1].axis('off')
    
    plt.tight_layout()
    plt.show()
    print()

### Compare Attention Across All Layers

In [ ]:
# Compare mean attention across all layers for one image
sample_img_path = sample_paths[0]
img = Image.open(sample_img_path).convert('RGB')
img_tensor = transform(img)

attentions = get_attention_maps(model, img_tensor, device)
num_layers = len(attentions)

fig, axes = plt.subplots(2, (num_layers + 1) // 2 + 1, figsize=(24, 8))
fig.suptitle(f'Attention Across All {num_layers} Layers (Mean over Heads)', 
            fontsize=16, fontweight='bold')

img_resized = img.resize((224, 224))

# Original image in the first cell
axes[0, 0].imshow(img_resized)
axes[0, 0].set_title('Original', fontsize=10, fontweight='bold')
axes[0, 0].axis('off')
axes[1, 0].axis('off')

for layer_idx in range(num_layers):
    row = layer_idx // ((num_layers + 1) // 2)
    col = layer_idx % ((num_layers + 1) // 2) + 1
    
    attn = attentions[layer_idx]
    cls_attn = attn[0, :, 0, 1:].mean(0)  # Mean over heads
    cls_attn = cls_attn.reshape(14, 14).numpy()
    cls_attn = (cls_attn - cls_attn.min()) / (cls_attn.max() - cls_attn.min() + 1e-8)
    
    attn_resized = np.array(Image.fromarray((cls_attn * 255).astype(np.uint8)).resize((224, 224), Image.BILINEAR)) / 255.0
    
    axes[row, col].imshow(img_resized)
    axes[row, col].imshow(attn_resized, cmap='hot', alpha=0.6)
    axes[row, col].set_title(f'Layer {layer_idx + 1}', fontsize=10)
    axes[row, col].axis('off')

# Hide the remaining empty cells
for ax in axes.flat:
    if not ax.get_title() and not ax.images:
        ax.axis('off')

plt.tight_layout()
plt.show()

print("\n💡 Observation: shallow layers attend to broad regions,")
print("   while deeper layers focus their attention on the object (the animal)!")

### Simple Foreground Segmentation Using Attention

In [ ]:
# Perform simple segmentation using DINO's last-layer attention
fig, axes = plt.subplots(len(sample_paths[:4]), 4, figsize=(16, 4 * len(sample_paths[:4])))
fig.suptitle('🎯 DINO Attention-based Foreground Segmentation', fontsize=16, fontweight='bold')

for row, img_path in enumerate(sample_paths[:4]):
    img = Image.open(img_path).convert('RGB').resize((224, 224))
    img_tensor = transform(Image.open(img_path).convert('RGB'))
    
    attentions = get_attention_maps(model, img_tensor, device)
    
    # Mean over all heads of the last layer
    cls_attn = attentions[-1][0, :, 0, 1:].mean(0).reshape(14, 14).numpy()
    
    # Normalize
    cls_attn = (cls_attn - cls_attn.min()) / (cls_attn.max() - cls_attn.min() + 1e-8)
    
    # Threshold for segmentation
    threshold = np.percentile(cls_attn, 60)
    mask = cls_attn > threshold
    mask_resized = np.array(Image.fromarray(mask.astype(np.uint8) * 255).resize((224, 224), Image.NEAREST)) / 255.0
    attn_resized = np.array(Image.fromarray((cls_attn * 255).astype(np.uint8)).resize((224, 224), Image.BILINEAR)) / 255.0
    
    # Masked image
    img_arr = np.array(img) / 255.0
    masked_img = img_arr * mask_resized[:, :, np.newaxis]
    
    # Plot
    axes[row, 0].imshow(img)
    axes[row, 0].set_title('Original', fontsize=11)
    axes[row, 0].axis('off')
    
    axes[row, 1].imshow(attn_resized, cmap='inferno')
    axes[row, 1].set_title('Attention Map', fontsize=11)
    axes[row, 1].axis('off')
    
    axes[row, 2].imshow(mask_resized, cmap='gray')
    axes[row, 2].set_title('Binary Mask', fontsize=11)
    axes[row, 2].axis('off')
    
    axes[row, 3].imshow(masked_img)
    axes[row, 3].set_title('Segmented', fontsize=11)
    axes[row, 3].axis('off')

plt.tight_layout()
plt.show()

print("\n💡 DINO's attention naturally captures object regions without any explicit supervision.")
print("   This is one of the key findings of the DINO paper!")

---
## 📝 Summary

What we confirmed in this practice:

| Experiment | Result |
|------|------|
| **Linear Probing** | High classification accuracy with frozen DINO features alone |
| **Cosine Similarity** | Intra-class similarity > inter-class similarity |
| **t-SNE** | Forms clear clusters for each class |
| **Attention Maps** | Naturally attends to object regions despite training without labels |

### Key Takeaways

1. **Power of self-supervised learning**: meaningful visual representations can be learned even without labels
2. **DINO's emergent properties**: attention naturally learns object segmentation
3. **Effectiveness of linear probing**: a simple way to evaluate the quality of pretrained representations

### Things to try next
- Compare `vit_small_patch16_224.dino` vs `vit_base_patch16_224.dino`
- Compare DINO vs DINOv2
- Extend to more classes (e.g., 10 animal species)
- Also evaluate with a KNN classifier

---
*This notebook was created for DINO ViT practice.*